# Nhóm 16 — P10-T05: Benchmark chính thức & Profiling NSYS cho V5 (Hai CUDA Stream)

Người thực hiện: **Lê Thượng Đế (23120232)**.

Notebook này thực hiện trọn gói yêu cầu **P10-T05** trên **Google Colab GPU Tesla T4 (`sm_75`)**:
1. Kiểm tra môi trường T4, clone đúng SHA đã tích hợp V5 (`main`), tải và xác thực SHA-256 cho 24 ảnh dataset cuối kỳ.
2. Build `build/stitch` bằng `nvcc -arch=sm_75 -std=c++11 -O2 --fmad=false` và chạy kiểm tra nhanh `verify_v5_cli.py`.
3. **Benchmark đồng phiên (không bật profiler, 1 warm-up + 5 runs):** đo **V0**, **V4 Production**, **V5 Sequential** (`--v5-schedule sequential`) và **V5 Dual** (`--v5-schedule dual`) kèm sidecar `--v5-detail-csv` (`PreS4_ms`, `Host_PreS4_ms`, `CountWait_A_ms`, `CountWait_B_ms`) trên 9 ca chuẩn + 3 ca `challenge`.
4. **5 khối đo ghép cặp luân phiên V4 / V5 Dual** trên ca chính `site2/large` (và đo lại mọi ca chuẩn có dấu hiệu chậm >5%).
5. **Profiling Nsight Systems (`nsys`) riêng biệt:** thu `.nsys-rep` + `.sqlite` kèm `--cuda-memory-usage=true` cho V4, V5 Sequential, V5 Dual trên `site1/large`, `site2/large`, `site3/large`; trích xuất **Peak VRAM**, **thời gian đến S4**, **độ chồng lấn giữa Stream A và Stream B**, vẽ biểu đồ timeline và sinh bản nháp báo cáo APOD.
6. Đóng gói toàn bộ bằng chứng thành `Group16_P10_T05_Evidence.zip` kèm SHA-256 để tải về và lưu Drive `v5/`.

In [ ]:
# @title 1. Cấu hình P10-T05
GIT_URL = "https://github.com/nh-atuan/panorama-stitching.git"  # @param {type:"string"}
GIT_REF = "26c8f241e056a7cf5e0c5bb0d85f036d9eafd9e1"  # @param {type:"string"}
RUNS = 5  # @param {type:"integer"}
RUN_PROFILING = True  # @param {type:"boolean"}
MOUNT_DRIVE = False  # @param {type:"boolean"}
DRIVE_RESULTS_DIR = "/content/drive/MyDrive/CSC14120_Group16_Results/v5"  # @param {type:"string"}
DATASET_ZIP = "/content/dataset.zip"  # @param {type:"string"}

print(f"GIT_REF={GIT_REF} | RUNS={RUNS} | RUN_PROFILING={RUN_PROFILING}")

In [ ]:
# @title 2. Khởi tạo workspace, kiểm tra Tesla T4 và clone mã nguồn
import csv
import hashlib
import json
import math
import os
import platform
import shutil
import sqlite3
import statistics
import subprocess
import sys
import zipfile
from datetime import datetime, timezone
from pathlib import Path

WORK = Path("/content/p10_t05_work")
if WORK.exists():
    shutil.rmtree(WORK)
WORK.mkdir(parents=True)

PROJECT = WORK / "source"
DATASET = WORK / "dataset"
BENCH = WORK / "benchmark"
SIDECAR = WORK / "sidecar"
STABILITY = WORK / "stability_blocks"
PROFILE = WORK / "profiling"
PLOTS = WORK / "plots"
LOGS = WORK / "logs"
for d in (BENCH, SIDECAR, STABILITY, PROFILE, PLOTS, LOGS):
    d.mkdir(parents=True, exist_ok=True)

CMD_LOG = []

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

def run_cmd(argv, label, cwd=None, required=True, timeout=900):
    argv_str = [str(x) for x in argv]
    print(f"[{label}] $ {' '.join(argv_str)}")
    t0 = datetime.now(timezone.utc)
    proc = subprocess.run(argv_str, cwd=str(cwd) if cwd else None,
                          capture_output=True, text=True, timeout=timeout)
    t1 = datetime.now(timezone.utc)
    rec = {
        "label": label,
        "argv": argv_str,
        "cwd": str(cwd) if cwd else os.getcwd(),
        "returncode": proc.returncode,
        "started_utc": t0.isoformat(),
        "ended_utc": t1.isoformat(),
        "elapsed_s": round((t1 - t0).total_seconds(), 3),
    }
    CMD_LOG.append(rec)
    (LOGS / f"{label}.stdout.txt").write_text(proc.stdout, encoding="utf-8")
    (LOGS / f"{label}.stderr.txt").write_text(proc.stderr, encoding="utf-8")
    if required and proc.returncode != 0:
        print(proc.stdout[-1500:])
        print(proc.stderr[-1500:])
        raise RuntimeError(f"Command '{label}' failed with exit code {proc.returncode}")
    return rec, proc.stdout, proc.stderr

_, gpu_out, _ = run_cmd(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"], "nvidia_smi")
print("GPU:", gpu_out.strip())
if "T4" not in gpu_out:
    raise RuntimeError(f"Expected Tesla T4 GPU, got: {gpu_out.strip()}")

_, nvcc_out, _ = run_cmd(["nvcc", "--version"], "nvcc_version")
_, nsys_out, _ = run_cmd(["nsys", "--version"], "nsys_version", required=False)
print(nvcc_out.strip().splitlines()[-1])
print("NSYS:", nsys_out.strip())

run_cmd(["git", "clone", GIT_URL, str(PROJECT)], "git_clone")
run_cmd(["git", "checkout", GIT_REF], "git_checkout", cwd=PROJECT)
_, resolved_sha, _ = run_cmd(["git", "rev-parse", "HEAD"], "git_rev_parse", cwd=PROJECT)
RESOLVED_SHA = resolved_sha.strip()
print("Checked out SHA:", RESOLVED_SHA)

if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")

In [ ]:
# @title 3. Tải và kiểm tra SHA-256 bộ dataset cuối kỳ (24 ảnh)
if (DATASET / "manifest.csv").is_file():
    print(f"Dataset already present at {DATASET}, skipping download.")
elif Path(DATASET_ZIP).is_file():
    print(f"Extracting local {DATASET_ZIP} ...")
    with zipfile.ZipFile(DATASET_ZIP, "r") as zf:
        zf.extractall(WORK)
    if not (DATASET / "manifest.csv").is_file():
        candidates = list(WORK.rglob("manifest.csv"))
        if candidates:
            DATASET = candidates[0].parent
else:
    run_cmd([sys.executable, "scripts/fetch_and_prepare_dataset.py", "--out", str(DATASET)],
            "fetch_dataset", cwd=PROJECT, timeout=900)

manifest_path = DATASET / "manifest.csv"
if not manifest_path.is_file():
    raise FileNotFoundError(f"Missing manifest.csv in {DATASET}")

manifest_sha = sha256_file(manifest_path)
with manifest_path.open("r", encoding="utf-8", newline="") as f:
    rows = list(csv.DictReader(f))
assert len(rows) == 24, f"Expected 24 images in manifest, got {len(rows)}"
for row in rows:
    img_path = DATASET / (row.get("file") or row["relative_path"])
    assert img_path.is_file(), f"Missing dataset image: {img_path}"
    actual = sha256_file(img_path)
    assert actual == row["sha256"], f"SHA mismatch for {row['relative_path']}"
print(f"Dataset verified: 24/24 images match SHA-256. manifest.csv SHA-256 = {manifest_sha}")

In [ ]:
# @title 4. Biên dịch build/stitch (sm_75) và kiểm chứng nhanh V5 CLI
run_cmd([sys.executable, "scripts/build.py", "--compiler", "nvcc", "--target", "stitch", "--require-t4"],
        "build_stitch", cwd=PROJECT, timeout=600)
STITCH = PROJECT / "build" / "stitch"
BINARY_SHA = sha256_file(STITCH)
print(f"Built {STITCH} | SHA-256 = {BINARY_SHA}")

# Kiểm chứng parity V4 vs V5 (sequential & dual) + sidecar 14 cột trước khi benchmark
SMOKE_OUT = WORK / "v5_cli_smoke"
run_cmd([sys.executable, "scripts/verify/verify_v5_cli.py",
         "--stitch", str(STITCH), "--dataset", str(DATASET),
         "--out", str(SMOKE_OUT), "--require-t4"],
        "verify_v5_cli_smoke", cwd=PROJECT, timeout=300)
print("verify_v5_cli.py PASSED!")

## 3. Benchmark đồng phiên (không bật profiler, 1 warm-up + 5 runs)

Đo cùng một phiên Tesla T4, cùng binary `build/stitch`, cùng dataset cho 4 cấu hình:
- `V0` (CPU Sequential)
- `V4` (Production Baseline GPU)
- `V5 Sequential` (`--version 5 --v5-schedule sequential --v5-detail-csv ...`)
- `V5 Dual` (`--version 5 --v5-schedule dual --v5-detail-csv ...`)

Trên 9 cấu hình chuẩn (`site1`, `site2`, `site3` × `small`, `medium`, `large`) và ghi riêng 3 cấu hình `challenge`.

In [ ]:
# @title 5. Chạy benchmark V0, V4, V5 Sequential và V5 Dual (không bật profiler)
STANDARD_SITES = ["site1", "site2", "site3"]
ALL_SITES = ["site1", "site2", "site3", "challenge"]
SIZES = ["small", "medium", "large"]

# 5a. Chạy run_benchmark.py chuẩn của repo cho V0, V4, V5 (dual mặc định)
RAW_BENCH = BENCH / "run_benchmark_v0_v4_v5"
run_cmd([sys.executable, "scripts/run_benchmark.py",
         "--stitch", str(STITCH), "--dataset", str(DATASET),
         "--versions", "0", "4", "5", "--runs", str(RUNS),
         "--include-challenge", "--out", str(RAW_BENCH)],
        "run_benchmark_v0_v4_v5", cwd=PROJECT, timeout=1800)

# 5b. Đo riêng V5 Sequential và V5 Dual kèm --v5-detail-csv (14 cột Pre-S4) trên cùng phiên
def parse_stitch_csv_avg_min(csv_text):
    rows = [r for r in csv.reader(csv_text.splitlines()) if r]
    header = rows[0]
    data = {}
    for row in rows[1:]:
        key = row[0].strip()
        if key in ("Average", "Minimum"):
            data[key] = {header[i]: float(row[i]) for i in range(1, len(header))}
    return data

def parse_sidecar_avg(sidecar_path):
    with sidecar_path.open("r", encoding="utf-8", newline="") as f:
        rows = list(csv.DictReader(f))
    cols = ["H2D_A_ms", "H2D_B_ms", "S1_A_ms", "S1_B_ms", "S2_A_ms", "S2_B_ms",
            "S3_A_ms", "S3_B_ms", "PreS4_ms", "Host_PreS4_ms", "CountWait_A_ms", "CountWait_B_ms"]
    return {c: statistics.mean(float(r[c]) for r in rows) for c in cols}

V5_MODES_RESULTS = {"sequential": {}, "dual": {}}
V5_FAILURES = []

for mode in ("sequential", "dual"):
    mode_dir = SIDECAR / f"v5_{mode}"
    mode_dir.mkdir(parents=True, exist_ok=True)
    for site in ALL_SITES:
        for size in SIZES:
            left = DATASET / site / f"{size}_left.jpg"
            right = DATASET / site / f"{size}_right.jpg"
            out_img = mode_dir / f"{site}_{size}_panorama.jpg"
            detail_csv = mode_dir / f"{site}_{size}_detail.csv"
            main_csv = mode_dir / f"{site}_{size}_main.csv"
            label = f"v5_{mode}_{site}_{size}"
            rec, stdout, stderr = run_cmd(
                [STITCH, left, right, out_img,
                 "--version", "5", "--v5-schedule", mode,
                 "--v5-detail-csv", detail_csv, "--runs", str(RUNS), "--debug"],
                label, cwd=PROJECT, required=(site != "challenge"), timeout=600
            )
            if rec["returncode"] != 0:
                V5_FAILURES.append({"mode": mode, "site": site, "size": size,
                                    "exit_code": rec["returncode"], "stderr": stderr.strip().splitlines()[-1] if stderr.strip() else ""})
                print(f"  -> Expected failure on {site}/{size} ({mode}): exit={rec['returncode']}")
                continue
            main_csv.write_text(stdout, encoding="utf-8")
            V5_MODES_RESULTS[mode][(site, size)] = {
                "main": parse_stitch_csv_avg_min(stdout),
                "sidecar": parse_sidecar_avg(detail_csv),
            }

print(f"Completed V5 sequential/dual detailed runs. Failures recorded: {len(V5_FAILURES)}")

In [ ]:
# @title 6. Tổng hợp bảng so sánh V0 / V4 / V5 Sequential / V5 Dual & Pre-S4
def load_version_summary(version_num):
    path = RAW_BENCH / f"v{version_num}" / "summary.csv"
    res = {}
    with path.open("r", encoding="utf-8", newline="") as f:
        for row in csv.DictReader(f):
            if row["stat"].strip().lower() == "average":
                res[(row["site"], row["size"])] = {k: float(row[k]) for k in row if k.endswith("_ms")}
    return res

V0_AVG = load_version_summary(0)
V4_AVG = load_version_summary(4)
V5_BENCH_AVG = load_version_summary(5)

def geomean(values):
    return math.exp(sum(math.log(v) for v in values) / len(values))

SUMMARY_ROWS = []
SUSPECT_CASES = []

for site in ALL_SITES:
    for size in SIZES:
        key = (site, size)
        if key not in V4_AVG or key not in V5_MODES_RESULTS["dual"]:
            continue
        v0_ms = V0_AVG[key]["Total_ms"]
        v4_ms = V4_AVG[key]["Total_ms"]
        v5_bench_ms = V5_BENCH_AVG[key]["Total_ms"]
        v5_seq_ms = V5_MODES_RESULTS["sequential"][key]["main"]["Average"]["Total_ms"]
        v5_dual_ms = V5_MODES_RESULTS["dual"][key]["main"]["Average"]["Total_ms"]
        pres4_seq = V5_MODES_RESULTS["sequential"][key]["sidecar"]["PreS4_ms"]
        pres4_dual = V5_MODES_RESULTS["dual"][key]["sidecar"]["PreS4_ms"]
        host_pres4_dual = V5_MODES_RESULTS["dual"][key]["sidecar"]["Host_PreS4_ms"]
        cw_a = V5_MODES_RESULTS["dual"][key]["sidecar"]["CountWait_A_ms"]
        cw_b = V5_MODES_RESULTS["dual"][key]["sidecar"]["CountWait_B_ms"]

        sp_v5_v0 = v0_ms / v5_dual_ms
        sp_v5_v4 = v4_ms / v5_dual_ms
        sp_v5_seq = v5_seq_ms / v5_dual_ms
        sp_pres4 = pres4_seq / pres4_dual if pres4_dual > 0 else float("nan")

        if site in STANDARD_SITES and sp_v5_v4 < 0.95:
            SUSPECT_CASES.append(key)

        SUMMARY_ROWS.append({
            "site": site, "size": size,
            "V0_ms": v0_ms, "V4_ms": v4_ms,
            "V5_bench_dual_ms": v5_bench_ms,
            "V5_seq_ms": v5_seq_ms, "V5_dual_ms": v5_dual_ms,
            "V5_dual_vs_V0": sp_v5_v0,
            "V5_dual_vs_V4": sp_v5_v4,
            "V5_dual_vs_V5_seq": sp_v5_seq,
            "PreS4_seq_ms": pres4_seq, "PreS4_dual_ms": pres4_dual,
            "PreS4_speedup": sp_pres4,
            "Host_PreS4_dual_ms": host_pres4_dual,
            "CountWait_A_ms": cw_a, "CountWait_B_ms": cw_b,
        })

summary_csv = BENCH / "p10_v5_summary.csv"
with summary_csv.open("w", encoding="utf-8", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=list(SUMMARY_ROWS[0].keys()))
    writer.writeheader()
    writer.writerows(SUMMARY_ROWS)

std_rows = [r for r in SUMMARY_ROWS if r["site"] in STANDARD_SITES]
GM_V5_V0 = geomean([r["V5_dual_vs_V0"] for r in std_rows])
GM_V5_V4 = geomean([r["V5_dual_vs_V4"] for r in std_rows])
GM_V5_SEQ = geomean([r["V5_dual_vs_V5_seq"] for r in std_rows])
GM_PRES4 = geomean([r["PreS4_speedup"] for r in std_rows])

print(f"{'Case':<16} {'V0 ms':>9} {'V4 ms':>8} {'V5_seq':>8} {'V5_dual':>8} {'V5/V0':>7} {'V5/V4':>7} {'Dual/Seq':>8} {'PreS4 seq->dual':>18} {'PreS4 x':>7}")
for r in SUMMARY_ROWS:
    print(f"{r['site']+'/'+r['size']:<16} {r['V0_ms']:9.3f} {r['V4_ms']:8.3f} {r['V5_seq_ms']:8.3f} {r['V5_dual_ms']:8.3f} "
          f"{r['V5_dual_vs_V0']:6.2f}x {r['V5_dual_vs_V4']:6.3f}x {r['V5_dual_vs_V5_seq']:7.3f}x "
          f"{r['PreS4_seq_ms']:7.3f}->{r['PreS4_dual_ms']:7.3f} {r['PreS4_speedup']:6.3f}x")
print("-" * 110)
print(f"9-case Geometric Mean: V5/V0 = {GM_V5_V0:.2f}x | V5/V4 = {GM_V5_V4:.4f}x | V5_dual/V5_seq = {GM_V5_SEQ:.4f}x | PreS4 = {GM_PRES4:.4f}x")
print(f"Standard cases regressing >5% (V5/V4 < 0.95): {SUSPECT_CASES}")

In [ ]:
# @title 7. 5 khối đo ghép cặp luân phiên V4 / V5 Dual tại ca chính site2/large (và đo lại ca nghi ngờ)
PAIRED_BLOCKS = []
primary_site, primary_size = "site2", "large"
left_p = DATASET / primary_site / f"{primary_size}_left.jpg"
right_p = DATASET / primary_site / f"{primary_size}_right.jpg"

for block_id in range(5):
    order = ("v4", "v5_dual") if block_id % 2 == 0 else ("v5_dual", "v4")
    block_rec = {"block": block_id + 1, "order": "->".join(order)}
    for tag in order:
        out_img = STABILITY / f"block{block_id+1}_{tag}.jpg"
        cmd = [STITCH, left_p, right_p, out_img, "--runs", str(RUNS), "--debug"]
        if tag == "v4":
            cmd += ["--version", "4"]
        else:
            detail = STABILITY / f"block{block_id+1}_v5_dual_detail.csv"
            cmd += ["--version", "5", "--v5-schedule", "dual", "--v5-detail-csv", detail]
        _, stdout, _ = run_cmd(cmd, f"paired_block{block_id+1}_{tag}", cwd=PROJECT, timeout=600)
        parsed = parse_stitch_csv_avg_min(stdout)
        block_rec[f"{tag}_total_ms"] = parsed["Average"]["Total_ms"]
        if tag == "v5_dual":
            sc = parse_sidecar_avg(STABILITY / f"block{block_id+1}_v5_dual_detail.csv")
            block_rec["v5_dual_pres4_ms"] = sc["PreS4_ms"]
    block_rec["speedup_v5_vs_v4"] = block_rec["v4_total_ms"] / block_rec["v5_dual_total_ms"]
    PAIRED_BLOCKS.append(block_rec)

block_speedups = [b["speedup_v5_vs_v4"] for b in PAIRED_BLOCKS]
MEDIAN_BLOCK_SPEEDUP = statistics.median(block_speedups)
FAVOR_V5_COUNT = sum(1 for x in block_speedups if x > 1.0)

with (STABILITY / "paired_blocks_site2_large.csv").open("w", encoding="utf-8", newline="") as f:
    w = csv.DictWriter(f, fieldnames=list(PAIRED_BLOCKS[0].keys()))
    w.writeheader()
    w.writerows(PAIRED_BLOCKS)

for b in PAIRED_BLOCKS:
    print(f"Block {b['block']} ({b['order']}): V4={b['v4_total_ms']:.3f} ms | V5_dual={b['v5_dual_total_ms']:.3f} ms | "
          f"V5/V4={b['speedup_v5_vs_v4']:.4f}x | PreS4={b['v5_dual_pres4_ms']:.3f} ms")
print(f"Median V5/V4 = {MEDIAN_BLOCK_SPEEDUP:.4f}x | Min-Max = {min(block_speedups):.4f}x - {max(block_speedups):.4f}x | "
      f"Favor V5 = {FAVOR_V5_COUNT}/5")

# Đo lại các ca nghi ngờ (nếu có ca chuẩn chậm >5%)
RECHECK_RESULTS = []
for site, size in SUSPECT_CASES:
    l_img, r_img = DATASET / site / f"{size}_left.jpg", DATASET / site / f"{size}_right.jpg"
    _, out4, _ = run_cmd([STITCH, l_img, r_img, STABILITY / f"recheck_{site}_{size}_v4.jpg",
                          "--version", "4", "--runs", str(RUNS)], f"recheck_{site}_{size}_v4", cwd=PROJECT)
    _, out5, _ = run_cmd([STITCH, l_img, r_img, STABILITY / f"recheck_{site}_{size}_v5.jpg",
                          "--version", "5", "--v5-schedule", "dual", "--runs", str(RUNS)], f"recheck_{site}_{size}_v5", cwd=PROJECT)
    t4 = parse_stitch_csv_avg_min(out4)["Average"]["Total_ms"]
    t5 = parse_stitch_csv_avg_min(out5)["Average"]["Total_ms"]
    RECHECK_RESULTS.append({"site": site, "size": size, "V4_ms": t4, "V5_dual_ms": t5, "V5_vs_V4": t4 / t5})
    print(f"Recheck {site}/{size}: V4={t4:.3f} ms, V5_dual={t5:.3f} ms, V5/V4={t4/t5:.4f}x")

## 4. Profiling Nsight Systems (`nsys`), Overlap Stream A/B & Peak VRAM

Thu trace **tách biệt hoàn toàn khỏi benchmark 5 runs**: chạy `--runs 1` (1 warm-up + 1 lượt đo) với `--trace=cuda,nvtx --cuda-memory-usage=true` cho **V4**, **V5 Sequential** và **V5 Dual** trên `site1/large`, `site2/large`, `site3/large`, sau đó xuất SQLite để đo:
- **Peak VRAM** (dynamic device allocations từ `CUDA_GPU_MEMORY_USAGE_EVENTS`),
- **Thời gian từ H2D đầu tiên đến khi S4 bắt đầu** (`GPU_H2D_to_first_S4_ms`),
- **Độ chồng lấn thực tế giữa các CUDA stream trước S4** (tổng thời lượng trên các stream riêng lẻ vs. hợp khoảng thời gian `union_duration`).

In [ ]:
# @title 8. Chạy nsys profile và xuất SQLite cho V4, V5 Sequential, V5 Dual (3 ca large)
PROFILE_CASES = [("site1", "large"), ("site2", "large"), ("site3", "large")]
PROFILE_MODES = [
    ("v4", ["--version", "4"]),
    ("v5_seq", ["--version", "5", "--v5-schedule", "sequential"]),
    ("v5_dual", ["--version", "5", "--v5-schedule", "dual"]),
]

NSYS_SQLITES = {}
if RUN_PROFILING:
    for tag, flags in PROFILE_MODES:
        for site, size in PROFILE_CASES:
            case_dir = PROFILE / tag / f"{site}_{size}"
            case_dir.mkdir(parents=True, exist_ok=True)
            rep_base = case_dir / "capture"
            sqlite_path = case_dir / "capture.sqlite"
            out_img = case_dir / "panorama.jpg"
            cmd = [
                "nsys", "profile",
                "--trace=cuda,nvtx",
                "--cuda-memory-usage=true",
                "--force-overwrite=true",
                "-o", str(rep_base),
                str(STITCH),
                str(DATASET / site / f"{size}_left.jpg"),
                str(DATASET / site / f"{size}_right.jpg"),
                str(out_img),
                *flags, "--runs", "1"
            ]
            run_cmd(cmd, f"nsys_{tag}_{site}_{size}", cwd=PROJECT, timeout=600)
            run_cmd(["nsys", "export", "--type=sqlite", "--force-overwrite=true",
                     "-o", str(sqlite_path), str(rep_base) + ".nsys-rep"],
                    f"nsys_export_{tag}_{site}_{size}", cwd=PROJECT, timeout=300)
            NSYS_SQLITES[(tag, site, size)] = sqlite_path
    print(f"Collected {len(NSYS_SQLITES)} NSYS traces + SQLite exports.")

In [ ]:
# @title 9. Phân tích SQLite NSYS: Peak VRAM, Pre-S4, S4 Start & Stream Overlap + Vẽ Timeline
import matplotlib.pyplot as plt

def union_duration(intervals):
    if not intervals:
        return 0
    merged_s, merged_e = None, None
    total = 0
    for s, e in sorted(intervals):
        if e <= s:
            continue
        if merged_s is None:
            merged_s, merged_e = s, e
        elif s <= merged_e:
            merged_e = max(merged_e, e)
        else:
            total += merged_e - merged_s
            merged_s, merged_e = s, e
    if merged_s is not None:
        total += merged_e - merged_s
    return total

def analyze_nsys_sqlite(sqlite_path):
    with sqlite3.connect(Path(sqlite_path).resolve().as_uri() + "?mode=ro", uri=True) as db:
        db.row_factory = sqlite3.Row
        strings = {r["id"]: r["value"] for r in db.execute("SELECT id, value FROM StringIds")}
        ranges = list(db.execute(
            "SELECT n.start, n.end, n.globalTid, coalesce(n.text, s.value) AS label "
            "FROM NVTX_EVENTS n LEFT JOIN StringIds s ON n.textId = s.id "
            "WHERE n.end IS NOT NULL ORDER BY n.start"
        ))
        runtime = {r["correlationId"]: dict(r) for r in db.execute(
            "SELECT r.*, s.value AS api FROM CUPTI_ACTIVITY_KIND_RUNTIME r "
            "JOIN StringIds s ON r.nameId = s.id"
        )}
        ops = []
        for table, kind in (("CUPTI_ACTIVITY_KIND_KERNEL", "kernel"),
                            ("CUPTI_ACTIVITY_KIND_MEMCPY", "memcpy"),
                            ("CUPTI_ACTIVITY_KIND_MEMSET", "memset")):
            for row in db.execute(f"SELECT * FROM {table}"):
                op = dict(row)
                api = runtime.get(op["correlationId"])
                if api:
                    op["api_start"] = api["start"]
                    op["api_end"] = api["end"]
                    op["tid"] = api["globalTid"]
                    op["kind"] = kind
                    if kind == "kernel":
                        op["name"] = strings.get(op.get("shortName") or op.get("demangledName"), "kernel")
                    elif kind == "memcpy":
                        op["name"] = f"memcpy(kind={op.get('copyKind')},B={op.get('bytes')})"
                    else:
                        op["name"] = f"memset(B={op.get('bytes')})"
                    ops.append(op)

        # Tìm lượt đo thứ 2 (sau warm-up) dựa trên NVTX S4.match
        s4_ranges = [r for r in ranges if r["label"] == "S4.match"]
        s4 = s4_ranges[-1]
        s4_ops = [op for op in ops if s4["start"] <= op["api_start"] <= s4["end"] and op["tid"] == s4["globalTid"]]
        first_s4_gpu = min(op["start"] for op in s4_ops)

        # Xác định 2 lượt H2D ảnh lớn của lượt đo thứ 2 (3840*2160*3 bytes)
        uploads = sorted(
            (op for op in ops if op["kind"] == "memcpy" and op.get("copyKind") == 1
             and op.get("bytes", 0) >= 3840 * 2160 * 3 and op["api_start"] < s4["start"]),
            key=lambda o: o["start"]
        )[-2:]
        t0_gpu = min(op["start"] for op in uploads)

        # Tất cả GPU ops nằm trong cửa sổ Pre-S4 [t0_gpu, first_s4_gpu)
        pres4_ops = [op for op in ops if op["end"] > t0_gpu and op["start"] < first_s4_gpu]
        by_stream = {}
        for op in pres4_ops:
            sid = op.get("streamId", -1)
            by_stream.setdefault(sid, []).append((max(op["start"], t0_gpu), min(op["end"], first_s4_gpu)))

        stream_busy_ns = {sid: union_duration(iv) for sid, iv in by_stream.items()}
        sum_streams_ns = sum(stream_busy_ns.values())
        union_all_ns = union_duration([(max(op["start"], t0_gpu), min(op["end"], first_s4_gpu)) for op in pres4_ops])
        overlap_ns = max(0, sum_streams_ns - union_all_ns)

        # Peak VRAM từ CUDA_GPU_MEMORY_USAGE_EVENTS (memKind=2: Device)
        allocs = {}
        peak_bytes = 0
        for row in db.execute("SELECT * FROM CUDA_GPU_MEMORY_USAGE_EVENTS WHERE memKind=2 ORDER BY start"):
            key = (row["contextId"], row["address"])
            if row["memoryOperationType"] == 0:
                allocs[key] = row["bytes"]
            elif row["memoryOperationType"] == 1:
                allocs.pop(key, None)
            peak_bytes = max(peak_bytes, sum(allocs.values()))

        return {
            "GPU_H2D_to_first_S4_ms": (first_s4_gpu - t0_gpu) / 1e6,
            "GPU_pres4_union_busy_ms": union_all_ns / 1e6,
            "GPU_pres4_sum_streams_ms": sum_streams_ns / 1e6,
            "GPU_pres4_overlap_ms": overlap_ns / 1e6,
            "GPU_pres4_overlap_pct": (100.0 * overlap_ns / union_all_ns) if union_all_ns > 0 else 0.0,
            "active_streams_before_S4": len(stream_busy_ns),
            "stream_busy_ms": {str(k): round(v / 1e6, 3) for k, v in sorted(stream_busy_ns.items())},
            "peak_vram_bytes": peak_bytes,
            "peak_vram_mib": round(peak_bytes / (1024 * 1024), 2),
            "remaining_vram_bytes": sum(allocs.values()),
            "t0_gpu": t0_gpu,
            "first_s4_gpu": first_s4_gpu,
            "pres4_ops": pres4_ops,
        }

NSYS_ANALYSIS = {}
for (tag, site, size), spath in NSYS_SQLITES.items():
    res = analyze_nsys_sqlite(spath)
    NSYS_ANALYSIS[f"{tag}:{site}/{size}"] = {k: v for k, v in res.items() if k not in ("pres4_ops", "t0_gpu", "first_s4_gpu")}
    print(f"{tag:7s} {site}/{size}: H2D->S4={res['GPU_H2D_to_first_S4_ms']:.3f} ms | "
          f"Overlap={res['GPU_pres4_overlap_ms']:.3f} ms ({res['GPU_pres4_overlap_pct']:.1f}%) | "
          f"Streams={res['active_streams_before_S4']} | PeakVRAM={res['peak_vram_mib']:.2f} MiB")

    if (site, size) == ("site2", "large"):
        NSYS_ANALYSIS[f"_plot_{tag}"] = res

with (PROFILE / "nsys_v5_analysis.json").open("w", encoding="utf-8") as f:
    clean_json = {k: v for k, v in NSYS_ANALYSIS.items() if not k.startswith("_plot_")}
    json.dump(clean_json, f, indent=2)

if all(f"_plot_{t}" in NSYS_ANALYSIS for t in ("v4", "v5_seq", "v5_dual")):
    fig, axes = plt.subplots(3, 1, figsize=(12, 6.5), sharex=True)
    titles = [("v4", "V4 Production (Single Stream)"),
              ("v5_seq", "V5 Sequential (--v5-schedule sequential)"),
              ("v5_dual", "V5 Dual-Stream (--v5-schedule dual)")]
    color_map = {"memcpy": "#1f77b4", "kernel": "#ff7f0e", "memset": "#2ca02c"}
    for ax, (tag, title) in zip(axes, titles):
        d = NSYS_ANALYSIS[f"_plot_{tag}"]
        t0 = d["t0_gpu"]
        streams = sorted({op.get("streamId", -1) for op in d["pres4_ops"]})
        y_map = {sid: idx for idx, sid in enumerate(streams)}
        for op in d["pres4_ops"]:
            s_ms = (max(op["start"], t0) - t0) / 1e6
            dur_ms = max(0.02, (min(op["end"], d["first_s4_gpu"]) - max(op["start"], t0)) / 1e6)
            y = y_map[op.get("streamId", -1)]
            ax.barh(y, dur_ms, left=s_ms, height=0.65, color=color_map.get(op["kind"], "gray"), alpha=0.85)
        s4_ms = (d["first_s4_gpu"] - t0) / 1e6
        ax.axvline(s4_ms, color="red", linestyle="--", linewidth=1.3, label=f"S4 start ({s4_ms:.2f} ms)")
        ax.set_yticks(range(len(streams)))
        ax.set_yticklabels([f"Stream {s}" for s in streams])
        ax.set_title(f"{title} — H2D->S4: {s4_ms:.2f} ms | Overlap: {d['GPU_pres4_overlap_ms']:.2f} ms ({d['GPU_pres4_overlap_pct']:.1f}%)")
        ax.legend(loc="upper right", fontsize=8)
        ax.grid(axis="x", linestyle=":", alpha=0.5)
    axes[-1].set_xlabel("Time from first measured H2D (ms) — site2/large (Tesla T4)")
    plt.tight_layout()
    fig.savefig(PLOTS / "nsys_v5_overlap_site2_large.png", dpi=180)
    plt.show()

In [ ]:
# @title 10. Vẽ biểu đồ Speedup & Pre-S4 và xuất bản nháp APOD P10-T05
import numpy as np

labels = [f"{r['site']}/{r['size']}" for r in std_rows]
x = np.arange(len(labels))

# Biểu đồ 1: Speedup V5 Dual vs V4 và V5 Dual vs V5 Seq
fig, ax = plt.subplots(figsize=(11, 4.2))
w = 0.36
ax.bar(x - w/2, [r["V5_dual_vs_V4"] for r in std_rows], w, label="V5 Dual / V4 Prod")
ax.bar(x + w/2, [r["V5_dual_vs_V5_seq"] for r in std_rows], w, label="V5 Dual / V5 Seq")
ax.axhline(1.0, color="black", linewidth=0.9)
ax.axhline(1.05, color="red", linestyle="--", linewidth=1.0, label="APOD 1.05x target")
ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=25, ha="right")
ax.set_ylabel("End-to-End Speedup (x)")
ax.set_title(f"P10-T05 V5 Speedup on Tesla T4 (GM V5/V4 = {GM_V5_V4:.4f}x, GM Dual/Seq = {GM_V5_SEQ:.4f}x)")
ax.legend()
ax.grid(axis="y", linestyle=":", alpha=0.5)
plt.tight_layout()
fig.savefig(PLOTS / "v5_speedup_vs_v4.png", dpi=180)
plt.show()

# Biểu đồ 2: Pre-S4 Sequential vs Dual (sidecar CSV)
fig, ax = plt.subplots(figsize=(11, 4.2))
ax.bar(x - w/2, [r["PreS4_seq_ms"] for r in std_rows], w, label="V5 Sequential Pre-S4 (ms)")
ax.bar(x + w/2, [r["PreS4_dual_ms"] for r in std_rows], w, label="V5 Dual Pre-S4 (ms)")
ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=25, ha="right")
ax.set_ylabel("Pre-S4 Duration (ms)")
ax.set_title(f"Pre-S4 Duration Comparison (9-case GM Pre-S4 Speedup = {GM_PRES4:.4f}x)")
ax.legend()
ax.grid(axis="y", linestyle=":", alpha=0.5)
plt.tight_layout()
fig.savefig(PLOTS / "v5_pres4_breakdown.png", dpi=180)
plt.show()

# Xuất bản nháp APOD từ số đo thật
p_row = [r for r in std_rows if r["site"] == "site2" and r["size"] == "large"][0]
keep_v5 = (MEDIAN_BLOCK_SPEEDUP >= 1.05 and FAVOR_V5_COUNT >= 4 and len(SUSPECT_CASES) == 0)
decision = "KEEP V5 AS PRODUCTION" if keep_v5 else "KEEP V4 AS PRODUCTION; RETAIN V5 AS CONCURRENCY CASE STUDY"

draft_lines = [
    "# P10-T05 Measured Summary (Generated on Tesla T4)",
    f"- Checked out SHA: `{RESOLVED_SHA}`",
    f"- Binary SHA-256: `{BINARY_SHA}`",
    f"- Dataset manifest SHA-256: `{manifest_sha}`",
    f"- 9-case GM V5_dual / V0: **{GM_V5_V0:.2f}x**",
    f"- 9-case GM V5_dual / V4: **{GM_V5_V4:.4f}x**",
    f"- 9-case GM V5_dual / V5_seq: **{GM_V5_SEQ:.4f}x**",
    f"- 9-case GM Pre-S4 speedup: **{GM_PRES4:.4f}x**",
    f"- Primary case `site2/large`: V4={p_row['V4_ms']:.3f} ms, V5_seq={p_row['V5_seq_ms']:.3f} ms, V5_dual={p_row['V5_dual_ms']:.3f} ms (V5/V4={p_row['V5_dual_vs_V4']:.4f}x, PreS4={p_row['PreS4_seq_ms']:.3f}->{p_row['PreS4_dual_ms']:.3f} ms, {p_row['PreS4_speedup']:.4f}x)",
    f"- 5 paired blocks (`site2/large`): median **{MEDIAN_BLOCK_SPEEDUP:.4f}x** (min-max {min(block_speedups):.4f}-{max(block_speedups):.4f}x), favor V5 = **{FAVOR_V5_COUNT}/5**",
    f"- APOD Decision: **{decision}**",
]
draft_text = "\n".join(draft_lines) + "\n"
(WORK / "p10_t05_apod_draft.md").write_text(draft_text, encoding="utf-8")
print(draft_text)

In [ ]:
# @title 11. Lưu environment.json, đóng gói ZIP kèm SHA-256 và tải xuống
env_info = {
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "git_ref_requested": GIT_REF,
    "git_sha_resolved": RESOLVED_SHA,
    "binary_sha256": BINARY_SHA,
    "dataset_manifest_sha256": manifest_sha,
    "gpu": gpu_out.strip(),
    "nvcc": nvcc_out.strip(),
    "nsys": nsys_out.strip(),
    "python": sys.version,
    "platform": platform.platform(),
    "runs_per_config": RUNS,
    "commands": CMD_LOG,
}
(WORK / "environment.json").write_text(json.dumps(env_info, indent=2), encoding="utf-8")

ZIP_PATH = Path("/content/Group16_P10_T05_Evidence.zip")
if ZIP_PATH.exists():
    ZIP_PATH.unlink()

include_dirs = [BENCH, SIDECAR, STABILITY, PROFILE, PLOTS, LOGS]
include_files = [WORK / "environment.json", WORK / "p10_t05_apod_draft.md"]

with zipfile.ZipFile(ZIP_PATH, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for fpath in include_files:
        if fpath.is_file():
            zf.write(fpath, fpath.relative_to(WORK))
    for folder in include_dirs:
        for item in sorted(folder.rglob("*")):
            if item.is_file() and (item.suffix not in (".jpg", ".png") or item.parent == PLOTS):
                zf.write(item, item.relative_to(WORK))

zip_sha = sha256_file(ZIP_PATH)
(Path("/content/Group16_P10_T05_Evidence.zip.sha256")).write_text(f"{zip_sha}  {ZIP_PATH.name}\n", encoding="utf-8")
print(f"Created {ZIP_PATH} ({ZIP_PATH.stat().st_size / (1024*1024):.2f} MiB) | SHA-256 = {zip_sha}")

if MOUNT_DRIVE and DRIVE_RESULTS_DIR:
    dst = Path(DRIVE_RESULTS_DIR)
    dst.mkdir(parents=True, exist_ok=True)
    shutil.copy2(ZIP_PATH, dst / ZIP_PATH.name)
    print(f"Copied ZIP to Drive: {dst / ZIP_PATH.name}")

try:
    from google.colab import files
    files.download(str(ZIP_PATH))
except Exception as e:
    print("Browser download skipped:", e)